# 07. KenLM + beam-search decoding

Self-contained notebook: loads a trained char-vocab checkpoint
(QuartzNet or compatible), builds a KenLM 4-gram LM from the
synthetic Russian number corpus, runs greedy / beam / N-best
decoding variants on dev, sweeps (alpha, beta), rescoring with an
FSA validator, and finally produces a test submission.

## 1. Setup

In [ ]:
# Idempotent data download guard — runs only if notebooks/data/ is
# empty or missing. Uncomment on Colab / Kaggle; local runs typically
# have data already unpacked.
from pathlib import Path as _P
_DATA = _P("notebooks/data")
if not _DATA.exists() or not any(_DATA.iterdir()):
    # import gdown, zipfile
    # gdown.download(
    #     url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
    #     output="/content/data.zip",
    #     quiet=False,
    #     fuzzy=True,
    # )
    # zipfile.ZipFile("/content/data.zip").extractall("notebooks/data")
    print("notebooks/data is empty — enable the gdown block above to fetch it.")
else:
    print(f"notebooks/data already populated: {_DATA}")

In [ ]:
# Env hardening MUST happen before `import torch` — otherwise the
# PYTORCH_CUDA_ALLOC_CONF setting is silently ignored.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# cudnn.benchmark=False — dev lengths vary, autotune churn is pure cost.
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("high")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

from gp1.data.collate import collate_fn
from gp1.data.dataset import SpokenNumbersDataset, preload_audio_cache
from gp1.data.manifest import records_from_csv
from gp1.decoding.beam_pyctc import BeamSearchConfig, BeamSearchDecoder
from gp1.decoding.greedy import greedy_decode
from gp1.features.melbanks import LogMelFilterBanks
from gp1.lm import build_synthetic_corpus, train_kenlm
from gp1.models.quartznet import QuartzNet10x4
from gp1.text.denormalize import (
    _ALL_KNOWN,
    fsa_constrained_best,
    safe_words_to_digits,
    words_to_digits,
)
from gp1.text.vocab import CharVocab
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.types import ManifestRecord

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

## 2. Config

In [ ]:
# ---- Fill in paths for your environment -------------------------------
# CKPT_PATH points to the concrete checkpoint file (.pt). Override to
# any QuartzNet/compatible checkpoint produced by 01_quartznet.ipynb
# (or another notebook) — the only constraint is that meta.json sits
# in the same directory.
CKPT_PATH = Path("checkpoints/01_quartznet/best.pt")

DATA_ROOT = Path("notebooks/data")
DEV_ROOT = DATA_ROOT / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "test"
DEV_CSV = DEV_ROOT / "dev.csv"

# Audio frontend settings — must match training. Adjust if your meta.json
# reports a different configuration.
SAMPLERATE = 16000
N_FFT = 512
N_MELS = 80
HOP_LENGTH = 160
WIN_LENGTH = 400
BATCH_SIZE = 32
DL_WORKERS = 2

META_PATH = CKPT_PATH.parent / "meta.json"
if META_PATH.exists():
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))
    print("Loaded meta.json:", {k: meta[k] for k in list(meta)[:4]})
else:
    meta = {"arch": "quartznet_10x4"}
    print("No meta.json next to checkpoint — defaulting to QuartzNet10x4.")

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
assert DEV_CSV.exists(), f"Dev CSV not found: {DEV_CSV}"

## 3. Load checkpoint

In [ ]:
# Vocab first — the CTC head size depends on it.
vocab = CharVocab()
print(f"vocab_size={vocab.vocab_size}, blank_id={vocab.blank_id}")

arch = meta.get("arch", "quartznet_10x4")
hparams = meta.get("hparams", {})

def _instantiate_model(arch: str, hparams: dict) -> torch.nn.Module:
    """Dispatch architecture name to a concrete encoder factory.

    Only QuartzNet is wired up here — other arches can be added by
    importing their class and branching below.
    """
    if arch.startswith("quartznet"):
        return QuartzNet10x4(
            vocab_size=vocab.vocab_size,
            d_model=int(hparams.get("d_model", 256)),
            dropout=float(hparams.get("dropout", 0.1)),
            subsample_factor=int(hparams.get("subsample_factor", 2)),
        )
    raise ValueError(
        f"Unsupported arch {arch!r} in 07_decode_kenlm.ipynb dispatcher. "
        f"Extend _instantiate_model to handle it."
    )

model = _instantiate_model(arch, hparams).to(device)

state = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
# torch.compile wraps the module in OptimizedModule; saved keys gain an
# "_orig_mod." prefix when somebody forgot to unwrap before torch.save.
if any(k.startswith("_orig_mod.") for k in state):
    state = {k.removeprefix("_orig_mod."): v for k, v in state.items()}

missing, unexpected = model.load_state_dict(state, strict=False)
if missing or unexpected:
    print(f"missing={missing[:5]}..., unexpected={unexpected[:5]}...")
else:
    print("Loaded checkpoint cleanly.")

model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=N_FFT,
    samplerate=SAMPLERATE,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
).to(device)
mel_extractor.eval()

## 4. Build vocab + dev DataLoader

In [ ]:
import random

dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Dev records: {len(dev_records)}")

AUDIO_CACHE = preload_audio_cache(dev_records, target_samplerate=SAMPLERATE)
# share_memory_() lets fork-spawned workers read zero-copy — big speedup.
for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()

def _worker_init(worker_id: int) -> None:
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)
    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)

dev_ds = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    augmenter=None,
    target_samplerate=SAMPLERATE,
    audio_cache=AUDIO_CACHE,
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=DL_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(DL_WORKERS > 0),
    prefetch_factor=2 if DL_WORKERS > 0 else None,
    worker_init_fn=_worker_init,
)

## 5. Forward pass on dev (cache log-probs)

In [ ]:
# Run the encoder once over the full dev set and cache per-sample
# (log_probs, output_lengths, transcription, spk_id). Downstream
# decoding variants iterate over this cache — no more forward passes.
cache: list[dict] = []
model.eval()
with torch.no_grad():
    for batch in dev_loader:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (
            (audio_lengths // HOP_LENGTH + 1)
            .clamp(max=mel.size(-1))
            .long()
        )
        out = model(mel, mel_lengths)
        log_probs_cpu = out.log_probs.detach().cpu()
        output_lengths_cpu = out.output_lengths.detach().cpu()
        for i in range(log_probs_cpu.size(0)):
            length = int(output_lengths_cpu[i].item())
            # Store a compact per-sample slice (avoid keeping padded frames).
            cache.append(
                {
                    "log_probs": log_probs_cpu[i, :length, :].clone(),
                    "output_length": length,
                    "transcription": batch.transcriptions[i],
                    "spk_id": batch.spk_ids[i],
                }
            )

print(f"Cached forward pass for {len(cache)} dev samples.")

## 6. Build KenLM

In [ ]:
lm_dir = Path("lm")
lm_dir.mkdir(exist_ok=True)
corpus_path = lm_dir / "corpus.txt"
lm_path = lm_dir / "lm.bin"

if not corpus_path.exists():
    build_synthetic_corpus(corpus_path, train_manifest=None)
    print(f"Corpus built: {corpus_path}")
else:
    print(f"Corpus already exists: {corpus_path}")

if not lm_path.exists():
    # train_kenlm expects (corpus_path, out_binary, order, vocab_limit_path).
    # Requires `lmplz` and `build_binary` on PATH — see gp1.lm docs.
    train_kenlm(corpus_path, lm_path, order=4, vocab_limit_path=None)
    print(f"LM built: {lm_path}")
else:
    print(f"LM already exists: {lm_path}")

## 7. Build lexicon (unigrams)

In [ ]:
# pyctcdecode accepts a flat unigram whitelist; we union the closed 42
# number-word vocabulary with every whitespace-separated token in the
# synthetic corpus (each line is a phrase like "сто тридцать два").
lexicon: set[str] = set(_ALL_KNOWN)

for line in corpus_path.read_text(encoding="utf-8").splitlines():
    for tok in line.split():
        lexicon.add(tok)

lexicon_list = sorted(lexicon)
print(f"Lexicon size: {len(lexicon_list)} unigrams (expected: 42)")
print("sample:", lexicon_list[:10])

## 8. Greedy baseline

In [ ]:
greedy_hyps: list[str] = []
for item in cache:
    # greedy_decode expects [B, T, V] + [B]; pass one-sample batches.
    lp = item["log_probs"].unsqueeze(0)
    ol = torch.tensor([item["output_length"]], dtype=torch.int64)
    greedy_hyps.append(greedy_decode(lp, ol, vocab)[0])

refs_digits = [item["transcription"] for item in cache]
spk_ids = [item["spk_id"] for item in cache]

greedy_digits = [safe_words_to_digits(h, fallback="") for h in greedy_hyps]
greedy_word_cer = compute_cer(refs_digits, greedy_hyps)
greedy_digit_cer = compute_cer(refs_digits, greedy_digits)
greedy_per_spk = compute_per_speaker_cer(refs_digits, greedy_digits, spk_ids)
greedy_max_spk_cer = max(greedy_per_spk.values()) if greedy_per_spk else 0.0

def _invalid_share(hyps: list[str]) -> float:
    n_invalid = 0
    for h in hyps:
        try:
            words_to_digits(h)
        except ValueError:
            n_invalid += 1
    return n_invalid / len(hyps) if hyps else 0.0

greedy_invalid = _invalid_share(greedy_hyps)
print(f"greedy word CER (on words): {greedy_word_cer:.4f}")
print(f"greedy digit CER: {greedy_digit_cer:.4f}")
print(f"greedy max-speaker digit CER: {greedy_max_spk_cer:.4f}")
print(f"greedy invalid share: {greedy_invalid:.4f}")

## 9. Beam baseline

In [ ]:
base_config = BeamSearchConfig(
    alpha=0.7,
    beta=1.0,
    beam_width=100,
    hotwords=("тысяча", "тысячи", "тысяч"),
    hotword_weight=8.0,
)
base_decoder = BeamSearchDecoder(
    vocab=vocab,
    kenlm_path=lm_path,
    unigrams=lexicon_list,
    config=base_config,
)

def _decode_all_with(decoder: BeamSearchDecoder) -> list[str]:
    hyps: list[str] = []
    for item in cache:
        lp = item["log_probs"].unsqueeze(0)
        ol = torch.tensor([item["output_length"]], dtype=torch.int64)
        hyps.append(decoder.decode_batch(lp, ol)[0])
    return hyps

beam_base_hyps = _decode_all_with(base_decoder)
beam_base_digits = [safe_words_to_digits(h, fallback="") for h in beam_base_hyps]
beam_base_word_cer = compute_cer(refs_digits, beam_base_hyps)
beam_base_digit_cer = compute_cer(refs_digits, beam_base_digits)
beam_base_per_spk = compute_per_speaker_cer(
    refs_digits, beam_base_digits, spk_ids
)
beam_base_max_spk = (
    max(beam_base_per_spk.values()) if beam_base_per_spk else 0.0
)
beam_base_invalid = _invalid_share(beam_base_hyps)
print(f"beam baseline digit CER: {beam_base_digit_cer:.4f}")
print(f"beam baseline invalid share: {beam_base_invalid:.4f}")

## 10. Alpha/beta grid search

In [ ]:
# For each (alpha, beta) we rebuild the pyctcdecode decoder and run it
# over all dev samples. pyctcdecode folds alpha/beta into the underlying
# beam scoring, so caching decoded beams across hyperparameters is
# unsafe — each pair must issue fresh decode_beams calls.
ALPHA_GRID = [0.3, 0.5, 0.7, 1.0, 1.3]
BETA_GRID = [0.0, 0.5, 1.0, 1.5]

grid_rows: list[dict] = []
best_row: dict | None = None

for alpha in ALPHA_GRID:
    for beta in BETA_GRID:
        cfg = BeamSearchConfig(
            alpha=alpha,
            beta=beta,
            beam_width=100,
            hotwords=base_config.hotwords,
            hotword_weight=base_config.hotword_weight,
        )
        dec = BeamSearchDecoder(
            vocab=vocab,
            kenlm_path=lm_path,
            unigrams=lexicon_list,
            config=cfg,
        )
        # OPTIMIZATION: call `_decoder.decode_beams` (pyctcdecode's
        # private API) directly. Public `decode_batch` only returns
        # the best beam, but the grid/rescoring flow needs the full
        # N-best list with scores, so we accept the private-attribute
        # access here. Safe within one (alpha, beta) pair; never
        # cache a decoder across pairs since pyctcdecode folds the
        # coefficients into internal scoring state.
        hyps: list[str] = []
        for item in cache:
            logits_np = item["log_probs"].float().numpy()
            beams = dec._decoder.decode_beams(
                logits_np,
                beam_width=cfg.beam_width,
                hotwords=list(cfg.hotwords),
                hotword_weight=cfg.hotword_weight,
            )
            hyps.append(beams[0][0] if beams else "")
        digits = [safe_words_to_digits(h, fallback="") for h in hyps]
        digit_cer = compute_cer(refs_digits, digits)
        row = {"alpha": alpha, "beta": beta, "digit_cer": digit_cer}
        grid_rows.append(row)
        if best_row is None or digit_cer < best_row["digit_cer"]:
            best_row = row
        print(
            f"alpha={alpha:.1f} beta={beta:.1f} digit_cer={digit_cer:.4f}"
        )

grid_df = pd.DataFrame(grid_rows).sort_values("digit_cer")
print(grid_df.to_string(index=False))
print(f"best (alpha, beta) = ({best_row['alpha']}, {best_row['beta']}) → CER {best_row['digit_cer']:.4f}")

## 11. N-best + FSA-validator rescoring

In [ ]:
# Rebuild decoder with best (alpha, beta) and run N-best decoding; then
# apply fsa_constrained_best to keep only beams that parse to a valid
# 4..6 digit number. Chain of fallbacks covers every failure mode so
# submissions never contain empty strings.
best_alpha = float(best_row["alpha"])
best_beta = float(best_row["beta"])
BEAM_WIDTH_NBEST = 100
N_KEEP = 10
LENGTH_RANGE = (4, 6)
FALLBACK_DIGITS = "0000"

best_cfg = BeamSearchConfig(
    alpha=best_alpha,
    beta=best_beta,
    beam_width=BEAM_WIDTH_NBEST,
    hotwords=base_config.hotwords,
    hotword_weight=base_config.hotword_weight,
)
best_decoder = BeamSearchDecoder(
    vocab=vocab,
    kenlm_path=lm_path,
    unigrams=lexicon_list,
    config=best_cfg,
)

rescored_digits: list[str] = []
best_beam_texts: list[str] = []
for idx, item in enumerate(cache):
    logits_np = item["log_probs"].float().numpy()
    # Private `_decoder.decode_beams` — public `decode_batch` drops
    # the N-best list we need for FSA rescoring. See cell 10 note.
    beams = best_decoder._decoder.decode_beams(
        logits_np,
        beam_width=BEAM_WIDTH_NBEST,
        hotwords=list(best_cfg.hotwords),
        hotword_weight=best_cfg.hotword_weight,
    )
    top_n = beams[:N_KEEP]
    top_text = top_n[0][0] if top_n else ""
    best_beam_texts.append(top_text)

    digits = fsa_constrained_best(top_n, length_range=LENGTH_RANGE)
    if not digits:
        # Fallback 1: best beam → safe_words_to_digits
        digits = safe_words_to_digits(top_text, fallback="")
    if not digits:
        # Fallback 2: greedy output → safe_words_to_digits
        digits = safe_words_to_digits(greedy_hyps[idx], fallback="")
    if not digits:
        digits = FALLBACK_DIGITS
    rescored_digits.append(digits)

rescored_digit_cer = compute_cer(refs_digits, rescored_digits)
rescored_per_spk = compute_per_speaker_cer(
    refs_digits, rescored_digits, spk_ids
)
rescored_max_spk = max(rescored_per_spk.values()) if rescored_per_spk else 0.0
rescored_word_cer = compute_cer(refs_digits, best_beam_texts)
rescored_invalid = _invalid_share(best_beam_texts)
print(f"beam+N-best+FSA digit CER: {rescored_digit_cer:.4f}")
print(f"beam+N-best+FSA max-speaker CER: {rescored_max_spk:.4f}")

## 12. Comparison table

In [ ]:
# Grid-best row: pick the (alpha, beta) combo with lowest digit CER and
# re-derive its full metric set from the already-cached beam outputs.
grid_best_cfg = BeamSearchConfig(
    alpha=best_alpha,
    beta=best_beta,
    beam_width=100,
    hotwords=base_config.hotwords,
    hotword_weight=base_config.hotword_weight,
)
grid_best_decoder = BeamSearchDecoder(
    vocab=vocab,
    kenlm_path=lm_path,
    unigrams=lexicon_list,
    config=grid_best_cfg,
)
grid_best_hyps = _decode_all_with(grid_best_decoder)
grid_best_digits = [safe_words_to_digits(h, fallback="") for h in grid_best_hyps]
grid_best_word_cer = compute_cer(refs_digits, grid_best_hyps)
grid_best_digit_cer = compute_cer(refs_digits, grid_best_digits)
grid_best_per_spk = compute_per_speaker_cer(
    refs_digits, grid_best_digits, spk_ids
)
grid_best_max_spk = (
    max(grid_best_per_spk.values()) if grid_best_per_spk else 0.0
)
grid_best_invalid = _invalid_share(grid_best_hyps)

comparison_rows = [
    {
        "method": "greedy",
        "word_cer": greedy_word_cer,
        "digit_cer": greedy_digit_cer,
        "max_spk_cer": greedy_max_spk_cer,
        "invalid_pct": greedy_invalid,
    },
    {
        "method": "beam baseline",
        "word_cer": beam_base_word_cer,
        "digit_cer": beam_base_digit_cer,
        "max_spk_cer": beam_base_max_spk,
        "invalid_pct": beam_base_invalid,
    },
    {
        "method": f"beam grid-best (a={best_alpha},b={best_beta})",
        "word_cer": grid_best_word_cer,
        "digit_cer": grid_best_digit_cer,
        "max_spk_cer": grid_best_max_spk,
        "invalid_pct": grid_best_invalid,
    },
    {
        "method": "beam+lexicon+N-best+FSA",
        "word_cer": rescored_word_cer,
        "digit_cer": rescored_digit_cer,
        "max_spk_cer": rescored_max_spk,
        "invalid_pct": rescored_invalid,
    },
]
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

## 13. Diagnostics

In [ ]:
# Top-10 worst dev examples for the best method (rescored digits).
from gp1.train.metrics import _edit_distance

worst = sorted(
    [
        {
            "idx": idx,
            "spk": spk_ids[idx],
            "ref": refs_digits[idx],
            "hyp": rescored_digits[idx],
            "edit": _edit_distance(refs_digits[idx], rescored_digits[idx]),
            "beam_text": best_beam_texts[idx],
            "greedy_text": greedy_hyps[idx],
        }
        for idx in range(len(cache))
    ],
    key=lambda r: r["edit"],
    reverse=True,
)[:10]

print("=== Top-10 worst examples (beam+N-best+FSA) ===")
for row in worst:
    print(
        f"[{row['idx']:>5}] spk={row['spk']:<10} edit={row['edit']:>3} "
        f"ref={row['ref']:>7} hyp={row['hyp']!r:<10}"
    )
    print(f"        greedy: {row['greedy_text']!r}")
    print(f"        beam:   {row['beam_text']!r}")

print("\n=== Per-speaker digit CER (beam+N-best+FSA) ===")
for spk in sorted(rescored_per_spk):
    print(f"  {spk:<15} {rescored_per_spk[spk]:.4f}")

## 14. Test submission

In [ ]:
# If notebooks/data/test/ is present, run the best decoding config on
# it and write notebooks/submission.csv. Build a ManifestRecord list
# from test.csv when available, otherwise fall back to filename globbing.
if TEST_ROOT is not None and TEST_ROOT.exists():
    test_csv = TEST_ROOT / "test.csv"
    if test_csv.exists():
        test_records = records_from_csv(test_csv, TEST_ROOT)
    else:
        import soundfile as sf
        globbed = sorted(
            list(TEST_ROOT.glob("*.wav")) + list(TEST_ROOT.glob("*.mp3"))
        )
        test_records = []
        for p in globbed:
            info = sf.info(str(p))
            test_records.append(
                ManifestRecord(
                    audio_path=p.resolve(),
                    transcription="0",  # dummy — not used for inference
                    spk_id="unknown",
                    gender="unknown",
                    ext=p.suffix.lstrip(".").lower(),
                    samplerate=info.samplerate,
                    duration_s=info.frames / info.samplerate,
                )
            )
    print(f"Test records: {len(test_records)}")

    test_ds = SpokenNumbersDataset(
        records=test_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=SAMPLERATE,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    predictions: list[tuple[str, str]] = []
    record_idx = 0
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // HOP_LENGTH + 1)
                .clamp(max=mel.size(-1))
                .long()
            )
            out = model(mel, mel_lengths)
            log_probs_cpu = out.log_probs.detach().cpu()
            output_lengths_cpu = out.output_lengths.detach().cpu()
            # Also run greedy once as a fallback source.
            greedy_batch = greedy_decode(
                log_probs_cpu, output_lengths_cpu, vocab
            )
            for i in range(log_probs_cpu.size(0)):
                length = int(output_lengths_cpu[i].item())
                logits_np = log_probs_cpu[i, :length, :].float().numpy()
                # Private decode_beams needed for N-best; see cell 10.
                beams = best_decoder._decoder.decode_beams(
                    logits_np,
                    beam_width=BEAM_WIDTH_NBEST,
                    hotwords=list(best_cfg.hotwords),
                    hotword_weight=best_cfg.hotword_weight,
                )
                top_n = beams[:N_KEEP]
                top_text = top_n[0][0] if top_n else ""
                digits = fsa_constrained_best(
                    top_n, length_range=LENGTH_RANGE
                )
                if not digits:
                    digits = safe_words_to_digits(top_text, fallback="")
                if not digits:
                    digits = safe_words_to_digits(
                        greedy_batch[i], fallback=""
                    )
                if not digits:
                    digits = FALLBACK_DIGITS
                rec = test_records[record_idx]
                record_idx += 1
                predictions.append((rec.audio_path.name, digits))

    import csv
    submission_path = Path("notebooks/submission.csv")
    submission_path.parent.mkdir(parents=True, exist_ok=True)
    with submission_path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["filename", "transcription"])
        writer.writerows(predictions)
    print(f"Submission written: {submission_path} ({len(predictions)} rows)")
else:
    print("TEST_ROOT not available — skipping submission.")